In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [162]:
metadata = pd.read_csv('combined_metadata.tsv', sep='\t', index_col=0)
# remove csf samples and two samples without age
metadata = metadata[metadata['type'] != 'CSF'].reset_index(drop=True)
metadata = metadata.drop(metadata.index[(metadata['age'] == 0) | (metadata['age'].isna())]).reset_index(drop=True)
metadata

,study,sample_id,subject_id,group,type,sex,age,apoe,apoe_carrier,apoe_dose
0,burgos_dbgap,01_16_SER,01_16,AD,serum,male,87.0,34.0,apoe4,apoe4
1,burgos_dbgap,01_18_SER,01_18,AD,serum,male,86.0,33.0,no_apoe4,no_apoe4
2,burgos_dbgap,01_37_SER,01_37,control,serum,female,88.0,34.0,apoe4,apoe4
3,burgos_dbgap,01_44_SER,01_44,control,serum,female,78.0,33.0,no_apoe4,no_apoe4
4,burgos_dbgap,01_46_SER,01_46,control,serum,female,90.0,33.0,no_apoe4,no_apoe4
...,...,...,...,...,...,...,...,...,...,...
574,toden,SRR10192327,2657,control,plasma,female,87.0,NaN,NaN,NaN
575,toden,SRR10192326,2658,control,plasma,male,86.0,NaN,NaN,NaN
576,toden,SRR10192325,2659,control,plasma,female,65.0,NaN,NaN,NaN
577,toden,SRR10192324,2660,control,plasma,male,66.0,NaN,NaN,NaN


In [ ]:
X = metadata.iloc[:,5:10].to_numpy()
X_encoded = OrdinalEncoder(categories=[['male', 'female'], ['no_apoe4', 'apoe4'],['no_apoe4', 'apoe4', 'apoe44']], handle_unknown = 'use_encoded_value', unknown_value = -1).fit_transform(X[:,[0,3,4]])
X = np.concatenate([X_encoded,X[:,1].reshape(-1,1)], axis=1)
y = metadata['group'].copy().to_numpy()
y[y=='control'] = 0
y[y=='AD'] = 1
y = y.astype(np.int32)
groups = metadata['study'].copy().to_numpy()

In [184]:
def runTrainTest(X, y, groups, model):
    param_dict = {'random_forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]}, 'naive_bayes': {'var_smoothing': [1e-9, 1e-8, 1e-7]}, 'logistic_regression': {'C': [0.01, 0.1, 1.0, 10.0]}}
    model_dict = {'random_forest': RandomForestClassifier(), 'naive_bayes': GaussianNB(), 'logistic_regression': LogisticRegression()}
    logo = LeaveOneGroupOut()
    accuracy = []
    recall = []
    precision = []
    f1 = []
    for train, test in logo.split(X,y,groups):
        X_train, X_test = X[train], X[test]
        y_train, y_test = y[train], y[test]
        
        train_groups = groups[train]
        train_cv = LeaveOneGroupOut()
        params = param_dict[model]
        
        cv = GridSearchCV(model_dict[model], params, cv = train_cv)
        cv.fit(X_train, y_train, groups = train_groups)

        y_pred = cv.predict(X_test)

        accuracy.append(accuracy_score(y_test, y_pred))
        recall.append(recall_score(y_test, y_pred))
        precision.append(precision_score(y_test, y_pred))
        f1.append(f1_score(y_test, y_pred))
        
    return accuracy, recall, precision, f1

In [185]:
accuracy, recall, precision, f1 = runTrainTest(X, y, groups, 'random_forest')
print(f'Average accuracy: {np.mean(accuracy)}')
print(f'Average recall: {np.mean(recall)}')
print(f'Average precision: {np.mean(precision)}')
print(f'Average f1 score: {np.mean(f1)}')

Average accuracy: 0.5337370562619385
Average recall: 0.4842584520003875
Average precision: 0.5489887744194554
Average f1 score: 0.5108430269923424
